# 12 高级设计模式：安全地调整颜色与效果

## 1. 本节点目标
在不破坏页面排列的前提下，为首页不同区域单独设置颜色、渐变、透明度、毛玻璃、边框、圆角、阴影和文字颜色。

## 2. 完成结果与验收
- 24 个首页区域可在固定缩略图中选中并即时预览。
- 右侧面板只控制背景、透明度、毛玻璃、边框、圆角、阴影和文字颜色。
- 位置、尺寸、字体字号与间距由程序统一控制，五张物品卡底边保持齐平。
- 图片框使用固定比例，有图、无图或原始图片比例不同都不会推动下方内容。
- 每页始终分配五个等宽槽位，不足五件时右侧留空而不是放大已有卡片。
- 首个账户是唯一页面管理员；普通账户看不到编辑入口，服务端也拒绝其保存请求。
- 保存后写入账户数据库，并转换为正式页面 CSS。
- 隔离账户完成真实页面验证，74 项自动测试通过。

## 3. 本节点文件结构
- `src/smart_laundry/visual_designer_component.py`：Streamlit v2 设计画布。
- `src/smart_laundry/visual_design.py`：默认布局、白名单校验和 CSS 转换。
- `src/smart_laundry/database.py`：创建 `visual_designs` 表。
- `src/smart_laundry/accounts.py`：按账户保存、读取和重置设计。
- `app.py`：打开统一的“外观设计”窗口并应用保存结果。

## 4. 关键代码解释
颜色和效果在浏览器内即时预览，避免每次调色都让 Python 页面重跑。只有点击保存时，JavaScript 才把最终 JSON 交给 Python；Python 丢弃任何布局字段，只校验并保存安全的颜色与效果参数。

In [ ]:
# 一个效果元素的精简示例
element = {
    'id': 'info', 'fill_type': 'glass',
    'color1': '#FFFFFF', 'color2': '#EEF2FF', 'opacity': 76, 'blur': 14
}
element

## 5. 数据流
1. Python 从 SQLite 读取当前账户设计。
2. Streamlit 将设计作为结构化数据传给浏览器画布。
3. 用户在浏览器本地选择区域并调整颜色与效果。
4. 点击保存后发送最终 JSON。
5. Python 白名单校验并保存。
6. `visual_design_css()` 将设计应用到正式页面。

## 6. 关键概念
- **自定义组件**：Streamlit 内嵌的 HTML、CSS 和 JavaScript 小应用。
- **固定布局**：位置、宽高和字号由程序管理，避免用户样式互相挤压。
- **即时预览**：调整控件时只更新浏览器中的缩略图。
- **白名单校验**：只接收系统认识的元素和安全参数。

## 7. 为什么这样设计
继续保留 Python、SQLite 和 Agent 后端，只把复杂交互放进 Streamlit 1.62 自带的 v2 组件。这样不需要安装 Node.js，也不需要把整个项目迁移为 React。

## 8. 常见错误与排查
- **卡片曾经错位**：新版会忽略旧设计中的坐标、尺寸与字号字段，可恢复默认效果。
- **四类卡片颜色相同**：在同一编辑器中分别选择衣物、床品、玩偶和宠物用品卡片。
- **无图片物品内容上移**：图片占位框与真实图片共用同一固定比例；若仍异常，先强制刷新页面。
- **末页卡片突然变宽**：分页网格固定创建五个槽位，空槽位应留在右侧。
- **调整未保留**：必须点击右上角“保存并应用”；成功后画布会关闭并显示提示。
- **毛玻璃不明显**：需要页面后方存在颜色层，并适当降低透明度。

## 9. 面试可能追问
**问：为什么不再开放位置和字号编辑？**
答：Streamlit 的组件高度会随内容变化，自由坐标和统一字号容易造成重叠、换行与卡片不齐；固定响应式布局更稳定。

**追问：如何避免任意 CSS 注入？**
答：数据库只保存数字、枚举和十六进制颜色，Python 会重建 CSS，不接受用户输入的完整 CSS。

## 10. 必须掌握的最少知识
能够解释浏览器负责预览、Python 只接收白名单效果参数、SQLite 负责持久化即可。无需背设计器的全部 JavaScript。

## 11. 可自测小题
1. 为什么颜色调整时不立即写数据库？
2. 毛玻璃由哪两个主要参数控制？
3. 为什么服务端必须丢弃位置和字号字段？

<details><summary>参考答案</summary>1. 减少重跑和无意义写入。2. 透明度与 blur。3. 防止旧数据或伪造请求再次破坏固定布局。</details>

## 12. 动手小练习
1. 把天气栏改为浅蓝毛玻璃，模糊设置为 18。
2. 将物品大方框改成灰粉到雾蓝的 135 度渐变。
3. 修改文字颜色后恢复默认效果，确认固定布局没有变化。

## 13. 本节点术语表
Canvas：设计预览画布；Inspector：右侧属性面板；Glassmorphism：毛玻璃风格；Whitelist：只允许指定字段；Trigger：保存时发送给 Python 的一次性事件。

## 14. 下一节点连接
下一步可以继续增加页面级背景效果和更多区域颜色，但仍保持同一套固定响应式布局。